In [1]:
import json
from pyspark.sql.functions import current_timestamp, input_file_name, lit
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType, TimestampType
)

SHORTCUT_PATH   = "Files/historical-transactions"
TARGET_TABLE    = "raw_historical_transactions"
WM_TABLE        = "_pipeline_watermarks"
LAYER_KEY       = "bronze_historical_files"
TS_FORMAT       = "yyyy-MM-dd HH:mm:ss"

schema = StructType([
    StructField("transaction_id",        StringType(),       False),
    StructField("account_id",            StringType(),       False),
    StructField("amount",                DecimalType(18, 2), True),
    StructField("merchant",              StringType(),       True),
    StructField("city",                  StringType(),       True),
    StructField("transaction_timestamp", TimestampType(),    True),
])
print("Config loaded.")

StatementMeta(, 48882876-6e50-4356-93a8-86b9b5f0e098, 3, Finished, Available, Finished, False)

Config loaded.


In [2]:
wm_row = (
    spark.table(WM_TABLE)
         .filter(f"layer_name = '{LAYER_KEY}'")
         .select("processed_files").collect()
)
processed_files = json.loads(wm_row[0]["processed_files"]) if wm_row else []
print(f"Already processed: {processed_files}")

StatementMeta(, 48882876-6e50-4356-93a8-86b9b5f0e098, 4, Finished, Available, Finished, False)

Already processed: ['transactions_2026_01.csv', 'transactions_2026_02.csv', 'transactions_2026_03.csv', 'transactions_2026_04.csv', 'transactions_2026_05.csv', 'transactions_2026_06.csv']


In [3]:
all_files = [f.name for f in notebookutils.fs.ls(SHORTCUT_PATH) if f.name.endswith(".csv")]
new_files = [f for f in all_files if f not in processed_files]

print(f"All files    : {all_files}")
print(f"New files    : {new_files}")

if not new_files:
    print("No new files. Exiting.")
    spark.stop()
    notebookutils.notebook.exit("NO_NEW_FILES")

StatementMeta(, 48882876-6e50-4356-93a8-86b9b5f0e098, 5, Finished, Available, Finished, False)

All files    : ['transactions_2026_01.csv', 'transactions_2026_02.csv', 'transactions_2026_03.csv', 'transactions_2026_04.csv', 'transactions_2026_05.csv', 'transactions_2026_06.csv']
New files    : []
No new files. Exiting.
ExitValue: NO_NEW_FILES

In [ ]:
new_paths = [f"{SHORTCUT_PATH}/{f}" for f in new_files]

df_new = (
    spark.read
         .option("header", True)
         .option("timestampFormat", TS_FORMAT)
         .schema(schema)
         .csv(new_paths)
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("source_file",         input_file_name())
         .withColumn("source_system",       lit("ADLS_HISTORICAL"))
)

row_count = df_new.count()
print(f"Rows to append: {row_count}")
df_new.show(5, truncate=False)

StatementMeta(, 48882876-6e50-4356-93a8-86b9b5f0e098, -1, Cancelled, , Cancelled, True)

In [ ]:
df_new.write.mode("append").format("delta").saveAsTable(TARGET_TABLE)
print(f"Appended {row_count} rows to {TARGET_TABLE}")

StatementMeta(, 48882876-6e50-4356-93a8-86b9b5f0e098, -1, Cancelled, , Cancelled, True)

In [ ]:
updated_list = json.dumps(processed_files + new_files)

spark.sql(f"""
    UPDATE {WM_TABLE}
    SET processed_files = '{updated_list}',
        rows_last_run   = {row_count},
        last_run_status = 'SUCCESS',
        last_run_at     = current_timestamp()
    WHERE layer_name = '{LAYER_KEY}'
""")
print(f"Watermark updated: {updated_list}")


StatementMeta(, 48882876-6e50-4356-93a8-86b9b5f0e098, -1, Cancelled, , Cancelled, True)